# Project — Labelled-data size vs. classical and transfer-learning traffic-sign classifiers

**COMS4036A Computer Vision · Honours project · presentations Tue 27 Oct (Week 14)**

One submission per group. Every member must be able to explain every cell, and the Week 14 Q&A checks that individually.

This notebook follows the layout of the labs: numbered sections, numbered questions (**Qx.y**), a code cell per question, an *Answer:* cell for every written question, W&B runs at the end and a submission checklist. The report (six pages max) is written from the answers here, and every number in it should be traceable to a cell in this notebook or a row in `results/results.csv`.

**The claim under test (from the proposal).** As labelled data decreases, an ImageNet-pretrained ResNet-18 that is fine-tuned keeps a higher macro-F1 than a classical MR8 filter-bank + linear pipeline, and the gap widens most sharply in the low-data regime.

**What is held fixed:** test set (GTSRB's official 12,630 images), class list (43), canonical image resolution, validation set, tuning budget, training subsets (the same nested subsets for every method at a given seed), evaluation code. **What varies:** training-set size $N \in \{200, 500, 1000, 2000, 5000\}$ and seed $\in \{0,1,2\}$.

| Section | What it does | GPU? |
|---|---|---|
| 0 | Setup, config, data cache | – |
| 1 | Inspect the data | – |
| 2 | The protocol: validation split, nested subsets | – |
| 3 | The metric: macro-F1 by hand, then against sklearn; majority baseline | – |
| 4 | Approach 1 — MR8 filter bank + linear classifier | light |
| 5 | Approach 2 — ResNet-18 fine-tuned (plus a frozen linear probe) | yes |
| 6 | Tuning, with the same budget for each method, on validation only | yes |
| 7 | The experiment: 5 sizes × 3 seeds × 4 methods, resumable | yes |
| 8 | Results: the data-efficiency curve | – |
| 9 | Cost: model size, latency, training time | yes |
| 10 | Where each method fails | – |
| 11 | W&B runs | – |
| 12 | Before you submit | – |

If time runs out: the proposal's reduced version is $N=1000$ alone. Section 7 runs $N=1000$ first for exactly that reason, so a partial run still gives a result.

## 0. GPU, W&B, and GTSRB

Open in [Colab](https://colab.research.google.com), **File ▸ Save a copy in Drive**, set **Runtime ▸ Change runtime type** to a GPU, and log in to W&B as in the labs. This project logs to a W&B project called `coms4036a-gtsrb`.

In [ ]:
# Group members — fill in before submitting.
MEMBERS = [
    ("Khuselo Sofohlo", "2729931"),
    ("Wakhiwe Ndzimandze", "2694083"),
    ("Niel Grobler", "2690034"),
]
for name, number in MEMBERS:
    print(f"{number}  {name}")

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

Every setting that an experiment depends on lives in `CFG`. W&B gets it as the run config, so each logged number carries the settings it came from. Change things here and nowhere else. `C`, `features` and `lr` start as `None` and are set from validation in Section 6. They are not hand-picked.

In [ ]:
CFG = dict(
    wandb_project="coms4036a-gtsrb",
    wandb_mode="online",          # "offline" if W&B is down, "disabled" to skip logging
    data_source="kaggle",         # "kaggle" (full 39,209-image release), "torchvision", or "local"
    kaggle_dataset="aakcodebreaker/german-traffic-sign-recognition-benchmark",
    data_dir=None,                # set this for data_source="local" (folder containing GTSRB/)
    data_root=".",                # download folder for data_source="torchvision"
    img_size=64,                  # canonical resolution every method starts from
    cnn_res=128,                  # ResNet input: the 64x64 image upsampled on the GPU
    n_classes=43,
    sizes=[200, 500, 1000, 2000, 5000],
    run_order=[1000, 200, 5000, 500, 2000],   # 1000 first: the reduced-version result
    seeds=[0, 1, 2],
    val_frac=0.10,                # fraction of *tracks* per class held out for validation
    split_seed=1234,              # fixes the validation split (not varied)
    tune_size=1000,               # tuning is done once, at this size, on validation
    # classical pipeline
    mr8_scales=[(0.5, 1.5), (1.0, 3.0), (2.0, 6.0)],   # (sigma across, sigma along)
    mr8_iso_sigma=2.0,
    grid=4,                       # spatial pooling grid (grid x grid cells)
    C_grid=[0.01, 0.1, 1.0, 10.0],
    feature_grid=["mr8", "full"],
    features=None, C=None, probe_C=None,     # chosen in Section 6
    # fine-tuning
    steps=1500,                   # same optimiser-step budget at every N
    batch=64,
    lr_grid=[1e-4, 3e-4, 1e-3],
    lr=None,                      # chosen in Section 6
    weight_decay=1e-4,
    eval_batch=512,
    results_csv="results/results.csv",
)

In [ ]:
import wandb
if CFG["wandb_mode"] == "online":
    wandb.login()

In [ ]:
import json
import os
import pathlib
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler
from torchvision.datasets import GTSRB
from torchvision.models import ResNet18_Weights, resnet18

DEV = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs("results/preds", exist_ok=True)
os.makedirs("figures", exist_ok=True)


def seed_everything(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(0)
print("device:", DEV, "· torch", torch.__version__)

The data is the [German Traffic Sign Recognition Benchmark](https://benchmark.ini.rub.de/) (Stallkamp et al., 2012): 43 classes of cropped sign photographs from video. The default source is the [Kaggle copy of the original release](https://www.kaggle.com/datasets/aakcodebreaker/german-traffic-sign-recognition-benchmark) (about 490 MB, Apache 2.0): `GTSRB/Final_Training/Images/000NN/*.ppm` (39,209 images) and `GTSRB/Final_Test/Images` with `GT-final_test.csv` (12,630 images). `kagglehub` downloads it. Public datasets usually need no login; if Colab asks, add `KAGGLE_USERNAME` and `KAGGLE_KEY` under Colab ▸ Secrets. `data_source="torchvision"` is a fallback; note that it downloads the smaller `GTSRB-Training_fixed` training archive. The loader finds the files by layout, so either source, or a local copy, works. Decoding every `.ppm` takes a minute or two, so the images are resized once to the canonical `img_size` and cached as one `.npz`. Later runs load the cache in seconds.

The training images come in **tracks**: about 30 consecutive video frames of one physical sign. The file name `TTTTT_FFFFF.ppm` is track and frame. The track id is kept because Section 2 needs it.

In [ ]:
CLASS_NAMES = [
    "20 km/h", "30 km/h", "50 km/h", "60 km/h", "70 km/h", "80 km/h", "end 80 km/h",
    "100 km/h", "120 km/h", "no passing", "no passing >3.5t", "right-of-way next int.",
    "priority road", "yield", "stop", "no vehicles", ">3.5t prohibited", "no entry",
    "general caution", "curve left", "curve right", "double curve", "bumpy road",
    "slippery road", "road narrows right", "road work", "traffic signals", "pedestrians",
    "children crossing", "bicycles crossing", "ice/snow", "wild animals",
    "end all limits", "turn right ahead", "turn left ahead", "ahead only",
    "straight or right", "straight or left", "keep right", "keep left", "roundabout",
    "end no passing", "end no passing >3.5t",
]
assert len(CLASS_NAMES) == CFG["n_classes"]


def find_samples():
    # returns (train, test) lists of (path, label), from whichever source CFG names
    if CFG["data_source"] == "torchvision":
        return (GTSRB(CFG["data_root"], split="train", download=True)._samples,
                GTSRB(CFG["data_root"], split="test", download=True)._samples)
    if CFG["data_source"] == "kaggle":
        import kagglehub
        root = pathlib.Path(kagglehub.dataset_download(CFG["kaggle_dataset"]))
    else:
        root = pathlib.Path(CFG["data_dir"])
    files = [p for p in root.rglob("*") if p.suffix.lower() in (".ppm", ".png", ".jpg")]
    is_test = lambda p: any("test" in part.lower() for part in p.relative_to(root).parts)
    # training: images inside a class folder named by its number (00000 ... 00042)
    train = sorted((str(p), int(p.parent.name)) for p in files
                   if not is_test(p) and p.parent.name.isdigit())
    # test: labels from GT-final_test.csv, matched to image files by name
    gt = next(p for p in root.rglob("*.csv") if p.name.lower() == "gt-final_test.csv")
    by_name = {p.name: p for p in files if is_test(p)}
    rows = pd.read_csv(gt, sep=";")
    test = [(str(by_name[f]), int(c)) for f, c in zip(rows.Filename, rows.ClassId)]
    print(f"found {len(train):,} training and {len(test):,} test images under {root}")
    return train, test


def load_split(samples, split):
    S = CFG["img_size"]
    n = len(samples)
    X = np.empty((n, S, S, 3), np.uint8)
    y = np.empty(n, np.int64)
    wh = np.empty((n, 2), np.int32)
    track = np.full(n, -1, np.int64)
    for i, (path, label) in enumerate(samples):
        im = Image.open(path).convert("RGB")
        wh[i] = im.size
        X[i] = np.asarray(im.resize((S, S), Image.BILINEAR))
        y[i] = label
        if split == "train":
            # globally unique track key: class * 10000 + track number from the file name
            track[i] = label * 10000 + int(os.path.basename(path).split("_")[0])
    return X, y, track, wh


cache = f"gtsrb{CFG['img_size']}.npz"
if os.path.exists(cache):
    z = np.load(cache)
    X_train, y_train, track_train, wh_train = z["X_train"], z["y_train"], z["track_train"], z["wh_train"]
    X_test, y_test, wh_test = z["X_test"], z["y_test"], z["wh_test"]
else:
    t0 = time.time()
    train_samples, test_samples = find_samples()
    X_train, y_train, track_train, wh_train = load_split(train_samples, "train")
    X_test, y_test, _, wh_test = load_split(test_samples, "test")
    np.savez(cache, X_train=X_train, y_train=y_train, track_train=track_train, wh_train=wh_train,
             X_test=X_test, y_test=y_test, wh_test=wh_test)
    print(f"decoded and cached in {time.time() - t0:.0f}s")
print("train", X_train.shape, "· test", X_test.shape)

## 1. Inspect the data

Look before measuring. Three things decide how to read every later number: how unbalanced the classes are, how many truly *independent* examples there are, and what resolution the images arrive at.

**Q1.1.** Report the number of training and test images, the number of tracks, and the minimum, median and maximum images per class in training. The proposal quotes 39,209 training and 12,630 test images; check that the counts match. Then say why the number of *tracks* matters more than the number of images for the claim this project makes.

In [ ]:
train_counts = np.bincount(y_train, minlength=CFG["n_classes"])
test_counts = np.bincount(y_test, minlength=CFG["n_classes"])
n_tracks = len(np.unique(track_train))
print(f"training images: {len(y_train):,} · test images: {len(y_test):,}")
print(f"tracks: {n_tracks:,} · mean images per track: {len(y_train) / n_tracks:.1f}")
print(f"images per class (train): min {train_counts.min()} ({CLASS_NAMES[train_counts.argmin()]}), "
      f"median {int(np.median(train_counts))}, max {train_counts.max()} ({CLASS_NAMES[train_counts.argmax()]})")
print(f"imbalance ratio max/min: {train_counts.max() / train_counts.min():.1f}")

*Answer:*

**Q1.2.** Plot the per-class counts for train and test on one set of axes (as fractions of each split, so the two can be compared). Plot the distribution of original image widths. Show one example per class in a grid.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
k = np.arange(CFG["n_classes"])
ax.bar(k - 0.2, train_counts / train_counts.sum(), 0.4, label="train")
ax.bar(k + 0.2, test_counts / test_counts.sum(), 0.4, label="test")
ax.set_xlabel("class id"); ax.set_ylabel("fraction of split"); ax.set_xticks(k)
ax.set_title("GTSRB class distribution"); ax.legend()
plt.tight_layout(); plt.savefig("figures/class_distribution.png", dpi=150); plt.show()

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(wh_train[:, 0], bins=50, alpha=0.7, label="train")
ax.hist(wh_test[:, 0], bins=50, alpha=0.7, label="test")
ax.axvline(CFG["img_size"], color="k", ls="--", label=f"canonical {CFG['img_size']} px")
ax.set_xlabel("original width (pixels)"); ax.set_ylabel("images"); ax.legend()
plt.tight_layout(); plt.savefig("figures/image_widths.png", dpi=150); plt.show()
print("median original width:", int(np.median(wh_train[:, 0])), "px;",
      f"{(wh_train[:, 0] < CFG['img_size']).mean():.0%} of training images are upsampled")

fig, axes = plt.subplots(4, 11, figsize=(16, 6.5))
for c, ax in zip(range(CFG["n_classes"]), axes.flat):
    ax.imshow(X_train[np.where(y_train == c)[0][len(np.where(y_train == c)[0]) // 2]])
    ax.set_title(f"{c}: {CLASS_NAMES[c]}", fontsize=6); ax.axis("off")
for ax in axes.flat[CFG["n_classes"]:]:
    ax.axis("off")
plt.tight_layout(); plt.savefig("figures/class_examples.png", dpi=150); plt.show()

**Q1.3.** Show the 30 frames of one track side by side. Then answer: if a training subset of 200 images happened to contain ten frames of the same track, how many independent examples is that? Why does that matter for a curve whose x-axis is "number of labelled images"? And why is a random image-level train/validation split on GTSRB optimistic?

In [ ]:
t = track_train[0]
frames = np.where(track_train == t)[0]
fig, axes = plt.subplots(1, len(frames), figsize=(len(frames) * 0.6, 1.2))
for ax, i in zip(np.atleast_1d(axes), frames):
    ax.imshow(X_train[i]); ax.axis("off")
plt.suptitle(f"one track ({len(frames)} frames, class {CLASS_NAMES[y_train[frames[0]]]})", fontsize=9)
plt.show()

*Answer:*

**Q1.4.** Signs such as *turn left ahead* / *turn right ahead*, *keep left* / *keep right* and *curve left* / *curve right* are mirror images of each other. What does that rule out as data augmentation, and what does it say about a feature that is invariant to orientation? Come back to this in Section 4.

*Answer:*

## 2. The protocol

The comparison is only as fair as its protocol, so the protocol is built and checked before any model sees an image.

- **Test:** the official 12,630-image test set. It is fixed, it is used once per trained model, and nothing is tuned on it.
- **Validation:** whole tracks, held out per class (`val_frac` of each class's tracks). It is used only in Section 6, at one training size, for both methods' hyperparameters.
- **Training pool:** everything that is not validation. For each seed, one set of **nested, class-stratified** subsets is drawn from it: the 200-image subset is inside the 500, which is inside the 1,000, and so on. Every method trains on *exactly the same images* at a given (size, seed).

Nesting means that moving along the curve only *adds* data. Otherwise each point would also be a different random draw, and the curve would mix two effects.

**Q2.1.** Implement `track_split` and `nested_subsets`. Stratification is proportional to the pool's class frequencies, with at least one image per class, so every subset contains all 43 classes. Print the actual size of each subset (rounding moves it a little off target) and the minimum images per class at each size.

In [ ]:
def track_split(y, track, frac, seed):
    rng = np.random.RandomState(seed)
    val = []
    for c in range(CFG["n_classes"]):
        tracks_c = np.unique(track[y == c])
        n_val = max(1, int(round(frac * len(tracks_c))))
        chosen = rng.choice(tracks_c, n_val, replace=False)
        val.append(np.where(np.isin(track, chosen))[0])
    val = np.sort(np.concatenate(val))
    pool = np.setdiff1d(np.arange(len(y)), val)
    return pool, val


def nested_subsets(pool, y, sizes, seed):
    rng = np.random.RandomState(seed)
    yp = y[pool]
    counts = np.bincount(yp, minlength=CFG["n_classes"])
    p = counts / counts.sum()
    perm = {c: rng.permutation(pool[yp == c]) for c in range(CFG["n_classes"])}
    out = {}
    for N in sorted(sizes):
        q = np.minimum(np.maximum(1, np.round(N * p).astype(int)), counts)
        out[N] = np.sort(np.concatenate([perm[c][:q[c]] for c in range(CFG["n_classes"])]))
    return out


POOL, VAL = track_split(y_train, track_train, CFG["val_frac"], CFG["split_seed"])
SUBSETS = {s: nested_subsets(POOL, y_train, CFG["sizes"], s) for s in CFG["seeds"]}
print(f"pool {len(POOL):,} · validation {len(VAL):,} images "
      f"({len(np.unique(track_train[VAL]))} tracks)")
for N in CFG["sizes"]:
    idx = SUBSETS[0][N]
    print(f"N={N:>5}: actual {len(idx):>5} · min/class {np.bincount(y_train[idx], minlength=43).min():>3}"
          f" · distinct tracks {len(np.unique(track_train[idx])):>4}")

**Q2.2.** Verify the protocol with assertions rather than by eye: every subset is nested in the next size up; every subset contains all 43 classes; no subset shares an image *or a track* with validation; for a fixed size, different seeds give different subsets; and the same seed gives the same subset when called again.

In [ ]:
for s in CFG["seeds"]:
    sz = sorted(CFG["sizes"])
    for a, b in zip(sz, sz[1:]):
        assert np.isin(SUBSETS[s][a], SUBSETS[s][b]).all(), f"seed {s}: {a} not inside {b}"
    for N in sz:
        idx = SUBSETS[s][N]
        assert len(np.unique(y_train[idx])) == CFG["n_classes"]
        assert not np.isin(idx, VAL).any()
        assert not np.isin(track_train[idx], track_train[VAL]).any()
assert not np.array_equal(SUBSETS[0][1000], SUBSETS[1][1000])
assert np.array_equal(nested_subsets(POOL, y_train, CFG["sizes"], 0)[1000], SUBSETS[0][1000])
json.dump({str(s): {str(N): SUBSETS[s][N].tolist() for N in CFG["sizes"]} for s in CFG["seeds"]},
          open("results/subsets.json", "w"))
print("protocol checks passed · subsets written to results/subsets.json")

**Q2.3.** The seed changes two things at once: *which images* are in the subset and *how the model is initialised and shuffled*. So the ± in Section 8 contains both. Is that what you want to report? What would you change to separate the two, and what would it cost?

*Answer:*

## 3. The metric

The primary metric is **macro-F1**: compute F1 per class, then take the unweighted mean over the 43 classes. Accuracy is secondary. The skills-ladder habit from Lab 4 applies here too: implement it yourself, check it on paper, then check it against the library.

**Q3.1.** On paper, for `y_true = [0,0,0,0,1,1,2,2,2,2]` and `y_pred = [0,0,0,1,1,2,2,2,2,0]`, give the per-class precision, recall and F1, the macro-F1, and the accuracy, as exact fractions.

*Answer:*

| class | TP | FP | FN | precision | recall | F1 |
|---|---|---|---|---|---|---|
| 0 | 3 | 1 | 1 | 3/4 | 3/4 | 3/4 |
| 1 | 1 | 1 | 1 | 1/2 | 1/2 | 1/2 |
| 2 | 3 | 1 | 1 | 3/4 | 3/4 | 3/4 |

Macro-F1 $= (3/4 + 1/2 + 3/4)/3 = 2/3 \approx 0.6667$. Accuracy $= 7/10 = 0.7$. Macro-F1 sits below accuracy because the small class (1) counts as much as the large ones.

**Q3.2.** Implement `macro_f1(y_true, y_pred)` from a confusion matrix built with `np.bincount` and no Python loop over samples. Check it against Q3.1 and against `sklearn.metrics.f1_score(average="macro")` on random predictions over 43 classes. Report the largest absolute difference.

In [ ]:
def confusion(y_true, y_pred, n=CFG["n_classes"]):
    return np.bincount(n * np.asarray(y_true) + np.asarray(y_pred), minlength=n * n).reshape(n, n)


def macro_f1(y_true, y_pred, n=CFG["n_classes"]):
    cm = confusion(y_true, y_pred, n)
    tp = np.diag(cm).astype(float)
    fp = cm.sum(0) - tp
    fn = cm.sum(1) - tp
    denom = 2 * tp + fp + fn
    f1 = np.divide(2 * tp, denom, out=np.zeros(n), where=denom > 0)
    return f1.mean(), f1


toy_t = [0, 0, 0, 0, 1, 1, 2, 2, 2, 2]
toy_p = [0, 0, 0, 1, 1, 2, 2, 2, 2, 0]
m, per = macro_f1(toy_t, toy_p, n=3)
print("toy macro-F1:", m, "per class:", per, "(expected 0.666…; 0.75, 0.5, 0.75)")
assert abs(m - 2 / 3) < 1e-12

rng = np.random.RandomState(0)
worst = 0.0
for _ in range(20):
    yt, yp = rng.randint(0, 43, 5000), rng.randint(0, 43, 5000)
    worst = max(worst, abs(macro_f1(yt, yp)[0] -
                           f1_score(yt, yp, average="macro", labels=np.arange(43), zero_division=0)))
print("largest |ours - sklearn| over 20 random trials:", worst)

**Q3.3.** The trivial floor. Predict the most frequent training class of the $N=1000$, seed-0 subset for every test image. Report its accuracy and macro-F1. The brief insists that no number stands without something simpler beside it. This is that something, and it shows how differently the two metrics treat a predictor that ignores the image.

In [ ]:
maj = np.bincount(y_train[SUBSETS[0][1000]], minlength=43).argmax()
pred_maj = np.full(len(y_test), maj)
print(f"majority class: {maj} ({CLASS_NAMES[maj]})")
print(f"accuracy {accuracy_score(y_test, pred_maj):.4f} · macro-F1 {macro_f1(y_test, pred_maj)[0]:.4f}")

*Answer:*

## 4. Approach 1 — MR8 filter bank + linear classifier

The MR8 bank (Varma & Zisserman, 2005) starts from 38 filters: an *edge* filter (first derivative of an anisotropic Gaussian) and a *bar* filter (second derivative) at 3 scales × 6 orientations, plus an isotropic Gaussian and a Laplacian of Gaussian. MR8 keeps only the **maximum response over the 6 orientations** at each scale, which gives 8 rotation-invariant maps. The bank was designed for textures. Signs are not textures, so the maps are pooled over a `grid × grid` spatial layout to keep *where* the structure is, then fed to a multinomial logistic regression.

The images here are 64 px, not the 200 px textures the bank was designed for, so the scales are smaller than Varma & Zisserman's (1,3), (2,6), (4,12). That is the one change made to the published bank, and it is in `CFG`.

**Q4.1.** Build the bank with zero-mean, unit-L1 kernels and plot all 38 filters.

In [ ]:
def gauss1d(sigma, x, order):
    g = np.exp(-x ** 2 / (2 * sigma ** 2)) / (np.sqrt(2 * np.pi) * sigma)
    if order == 0:
        return g
    if order == 1:
        return -x / sigma ** 2 * g
    return (x ** 2 - sigma ** 2) / sigma ** 4 * g


def make_bank(scales, iso_sigma, n_orient=6):
    R = int(np.ceil(3 * max([max(s) for s in scales] + [iso_sigma])))
    yy, xx = np.mgrid[-R:R + 1, -R:R + 1].astype(float)
    edges, bars = [], []
    for s_across, s_along in scales:
        for k in range(n_orient):
            th = np.pi * k / n_orient
            u = np.cos(th) * xx + np.sin(th) * yy        # along the filter
            v = -np.sin(th) * xx + np.cos(th) * yy       # across the filter
            edges.append(gauss1d(s_along, u, 0) * gauss1d(s_across, v, 1))
            bars.append(gauss1d(s_along, u, 0) * gauss1d(s_across, v, 2))
    r2 = xx ** 2 + yy ** 2
    gauss = np.exp(-r2 / (2 * iso_sigma ** 2))
    log = (r2 - 2 * iso_sigma ** 2) / iso_sigma ** 4 * gauss
    bank = np.stack(edges + bars + [gauss, log])
    for i in range(len(bank)):
        if i != len(bank) - 2:                           # the Gaussian keeps its mean (brightness)
            bank[i] -= bank[i].mean()
        bank[i] /= np.abs(bank[i]).sum()
    return bank.astype(np.float32), R


BANK, R = make_bank(CFG["mr8_scales"], CFG["mr8_iso_sigma"])
print("bank", BANK.shape)
fig, axes = plt.subplots(4, 10, figsize=(12, 5))
for ax in axes.flat:
    ax.axis("off")
for i, (ax, f) in enumerate(zip(axes.flat, BANK)):
    ax.imshow(f, cmap="RdBu_r"); ax.set_title(str(i), fontsize=7)
plt.suptitle("rows 0–17 edge, 18–35 bar (3 scales × 6 orientations), 36 Gaussian, 37 LoG")
plt.tight_layout(); plt.savefig("figures/filter_bank.png", dpi=150); plt.show()

**Q4.2.** Implement `filter_features(X, mode)`. Convert to grayscale and standardise each image (zero mean, unit standard deviation) so that brightness and contrast differences between tracks do not dominate. Convolve with the bank using reflect padding. For `mode="mr8"`, take the max of the absolute response over orientations, giving 8 maps. For `mode="full"`, keep all 38 (orientation is kept). Pool each map over a `grid × grid` layout, both mean |response| and mean signed response. Features depend only on images and never on labels, so they are computed once for the whole training set and the test set, then indexed by subset.

In [ ]:
GRAY_W = torch.tensor([0.299, 0.587, 0.114]).view(1, 3, 1, 1)


@torch.no_grad()
def filter_features(X, mode, batch=2048, dev=DEV):
    bank, gray = torch.from_numpy(BANK)[:, None].to(dev), GRAY_W.to(dev)
    out = []
    for i in range(0, len(X), batch):
        x = torch.from_numpy(X[i:i + batch]).to(dev).permute(0, 3, 1, 2).float() / 255
        g = (x * gray).sum(1, keepdim=True)
        g = (g - g.mean((2, 3), keepdim=True)) / (g.std((2, 3), keepdim=True) + 1e-6)
        r = F.conv2d(F.pad(g, (R, R, R, R), mode="reflect"), bank)            # (B, 38, S, S)
        if mode == "mr8":
            B, _, S, _ = r.shape
            e = r[:, :18].abs().view(B, 3, 6, S, S).amax(2)
            b = r[:, 18:36].abs().view(B, 3, 6, S, S).amax(2)
            r = torch.cat([e, b, r[:, 36:]], 1)                                # (B, 8, S, S)
        G = CFG["grid"]
        f = torch.cat([F.adaptive_avg_pool2d(r.abs(), G), F.adaptive_avg_pool2d(r, G)], 1)
        out.append(f.flatten(1).cpu())
    return torch.cat(out).numpy()


t0 = time.time()
FEATS = {m: (filter_features(X_train, m), filter_features(X_test, m)) for m in CFG["feature_grid"]}
for m, (a, b) in FEATS.items():
    print(f"{m}: train {a.shape} test {b.shape}")
print(f"{time.time() - t0:.1f}s")

**Q4.3.** `fit_linear(F, y, C)` standardises features with statistics from the *training subset only*, then fits a multinomial logistic regression. Train on $N=1000$, seed 0, with `mr8` features and $C=1$, and report validation macro-F1. The test set is not touched here.

In [ ]:
def fit_linear(Ftr, ytr, C, seed=0):
    scaler = StandardScaler().fit(Ftr)
    clf = LogisticRegression(C=C, max_iter=3000, random_state=seed)
    clf.fit(scaler.transform(Ftr), ytr)
    return lambda Fx: clf.predict(scaler.transform(Fx)), clf


idx = SUBSETS[0][1000]
t0 = time.time()
predict, _ = fit_linear(FEATS["mr8"][0][idx], y_train[idx], C=1.0)
print(f"MR8 + logistic, N=1000 seed 0: val macro-F1 "
      f"{macro_f1(y_train[VAL], predict(FEATS['mr8'][0][VAL]))[0]:.4f} · {time.time() - t0:.1f}s")

**Q4.4.** Is this a fair baseline or a straw man? Name two things that could make the classical pipeline look worse than it needs to be, and say which of them `feature_grid` and `C_grid` (tuned in Section 6) address.

*Answer:*

## 5. Approach 2 — ResNet-18, fine-tuned

ImageNet-pretrained ResNet-18 (`IMAGENET1K_V1`). The 1000-way head is replaced by a 43-way linear layer and *all* weights are fine-tuned with AdamW and a cosine schedule. It reads the same 64 × 64 canonical image as the classical pipeline, upsampled to `cnn_res` on the GPU and normalised with ImageNet statistics.

**Fixed budget, not fixed epochs.** Every run gets `steps` optimiser steps at batch `batch`, whatever $N$ is. With fixed epochs, a 5,000-image run would get 25× the compute of a 200-image run, and the curve would partly measure compute. With fixed steps, the small-$N$ runs go through their data many more times, which is the regime the question is about.

**No augmentation.** The classical pipeline sees each training image exactly as it is, and so does the CNN. Augmentation would be a second, unmeasured source of extra data. If there is time, it is an obvious extension, run as a separate method so that it does not change this one.

A second, cheap method is added beside it: a **linear probe**. The ImageNet ResNet-18 is frozen, its 512-d pooled features are extracted once, and the same `fit_linear` as Section 4 is fitted on top. It separates two questions that fine-tuning answers together: *are ImageNet features better?* and *does adapting them help?*

**Q5.1.** Implement `make_resnet`, `train_cnn` and `predict_cnn`. Fine-tune on $N=1000$, seed 0, at $lr = 3\times10^{-4}$. Plot the training loss against step, and report validation macro-F1 and wall-clock time.

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEV).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=DEV).view(1, 3, 1, 1)
X_train_t = torch.from_numpy(X_train)
X_test_t = torch.from_numpy(X_test)


def to_input(xb):
    x = xb.to(DEV, non_blocking=True).permute(0, 3, 1, 2).float() / 255
    x = F.interpolate(x, size=CFG["cnn_res"], mode="bilinear", align_corners=False)
    return (x - MEAN) / STD


def make_resnet(n_out=CFG["n_classes"]):
    m = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    m.fc = nn.Linear(m.fc.in_features, n_out)
    return m.to(DEV)


def batches(idx, bs, seed):
    # endless epoch-shuffled stream of index batches, reproducible from the seed
    g = np.random.RandomState(seed)
    while True:
        p = g.permutation(idx)
        for i in range(0, len(p) - bs + 1, bs):
            yield p[i:i + bs]


def train_cnn(idx, seed, lr, steps=None, log=None):
    steps = steps or CFG["steps"]
    seed_everything(seed)
    model = make_resnet()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=CFG["weight_decay"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps)
    scaler = torch.amp.GradScaler("cuda", enabled=DEV == "cuda")
    bs = min(CFG["batch"], len(idx))
    model.train()
    losses = []
    for step, b in zip(range(steps), batches(idx, bs, seed)):
        x = to_input(X_train_t[b])
        t = torch.from_numpy(y_train[b]).to(DEV)
        with torch.autocast(DEV, dtype=torch.float16, enabled=DEV == "cuda"):
            loss = F.cross_entropy(model(x), t)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt); scaler.update(); sched.step()
        losses.append(loss.item())
        if log is not None and step % 25 == 0:
            log({"train/loss": loss.item(), "train/lr": sched.get_last_lr()[0], "step": step})
    return model, losses


@torch.no_grad()
def predict_cnn(model, Xt):
    model.eval()
    out = []
    for i in range(0, len(Xt), CFG["eval_batch"]):
        with torch.autocast(DEV, dtype=torch.float16, enabled=DEV == "cuda"):
            out.append(model(to_input(Xt[i:i + CFG["eval_batch"]])).argmax(1).cpu())
    return torch.cat(out).numpy()


idx = SUBSETS[0][1000]
t0 = time.time()
model, losses = train_cnn(idx, seed=0, lr=3e-4)
t_train = time.time() - t0
print(f"fine-tune N=1000 seed 0: val macro-F1 "
      f"{macro_f1(y_train[VAL], predict_cnn(model, X_train_t[VAL]))[0]:.4f} · {t_train:.0f}s")
plt.figure(figsize=(6, 3.5))
plt.plot(losses); plt.yscale("log"); plt.xlabel("optimiser step"); plt.ylabel("training cross-entropy")
plt.title("ResNet-18 fine-tune, N=1000, seed 0"); plt.tight_layout(); plt.show()
del model

**Q5.2.** Extract frozen ImageNet ResNet-18 features (the 512-d output of global average pooling) for every training and test image, once. Fit `fit_linear` on $N=1000$, seed 0, and report validation macro-F1.

In [ ]:
@torch.no_grad()
def frozen_features(Xt):
    m = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1).to(DEV).eval()
    m.fc = nn.Identity()
    out = []
    for i in range(0, len(Xt), CFG["eval_batch"]):
        with torch.autocast(DEV, dtype=torch.float16, enabled=DEV == "cuda"):
            out.append(m(to_input(Xt[i:i + CFG["eval_batch"]])).float().cpu())
    return torch.cat(out).numpy()


t0 = time.time()
PROBE = (frozen_features(X_train_t), frozen_features(X_test_t))
print("probe features", PROBE[0].shape, f"· {time.time() - t0:.0f}s")
idx = SUBSETS[0][1000]
predict, _ = fit_linear(PROBE[0][idx], y_train[idx], C=1.0)
print(f"linear probe N=1000 seed 0: val macro-F1 {macro_f1(y_train[VAL], predict(PROBE[0][VAL]))[0]:.4f}")

**Q5.3.** The training loss at $N=200$ will go to nearly zero within a few hundred steps. Is that a problem for the comparison? What does the fixed step budget do to the 200-image runs that a fixed epoch budget would not?

*Answer:*

## 6. Tuning, with the same budget for each method

Each method gets a handful of hyperparameter values, tried **once**, at $N=$`tune_size` with seed 0, scored on the **validation** set. The winner is then frozen for every size and seed. Two things follow from this. Nothing is tuned on test. And the low-$N$ runs are not quietly helped by a validation set much larger than their training set, which would happen if each size were tuned separately.

| method | grid | fits |
|---|---|---|
| classical | `feature_grid` × `C_grid` | 8 |
| linear probe | `C_grid` | 4 |
| fine-tune | `lr_grid` | 3 |

**Q6.1.** Run the three grids, print validation macro-F1 for every value, and write the winners into `CFG`. Log the tuning table to W&B as one run.

In [ ]:
idx = SUBSETS[0][CFG["tune_size"]]
tune_rows = []

for feat in CFG["feature_grid"]:
    for C in CFG["C_grid"]:
        predict, _ = fit_linear(FEATS[feat][0][idx], y_train[idx], C)
        tune_rows.append(dict(method="classical", features=feat, C=C,
                              val_macro_f1=macro_f1(y_train[VAL], predict(FEATS[feat][0][VAL]))[0]))
for C in CFG["C_grid"]:
    predict, _ = fit_linear(PROBE[0][idx], y_train[idx], C)
    tune_rows.append(dict(method="probe", C=C,
                          val_macro_f1=macro_f1(y_train[VAL], predict(PROBE[0][VAL]))[0]))
for lr in CFG["lr_grid"]:
    model, _ = train_cnn(idx, seed=0, lr=lr)
    tune_rows.append(dict(method="finetune", lr=lr,
                          val_macro_f1=macro_f1(y_train[VAL], predict_cnn(model, X_train_t[VAL]))[0]))
    del model

TUNE = pd.DataFrame(tune_rows)
best = lambda m: TUNE[TUNE.method == m].sort_values("val_macro_f1", ascending=False).iloc[0]
CFG["features"], CFG["C"] = best("classical")["features"], float(best("classical")["C"])
CFG["probe_C"] = float(best("probe")["C"])
CFG["lr"] = float(best("finetune")["lr"])
TUNE.to_csv("results/tuning.csv", index=False)
print(TUNE.to_string(index=False))
print("chosen:", {k: CFG[k] for k in ["features", "C", "probe_C", "lr"]})

run = wandb.init(project=CFG["wandb_project"], name=f"tuning-n{CFG['tune_size']}",
                 config=CFG, mode=CFG["wandb_mode"], job_type="tuning")
wandb.log({"tuning": wandb.Table(dataframe=TUNE.astype(str))})
run.finish()

**Q6.2.** Did `full` (orientation kept) beat `mr8` (orientation discarded)? Connect the answer to Q1.4. If the best value of any grid sits at the edge of its grid, say so and say what it implies.

*Answer:*

## 7. The experiment

Four methods × five sizes × three seeds = 60 runs. Only the 15 fine-tuning runs need the GPU for long, about 1–2 minutes each on a T4. Every run:

1. trains on `SUBSETS[seed][N]` with the hyperparameters frozen in Section 6,
2. predicts the full test set once,
3. appends a row to `results/results.csv` and saves its test predictions to `results/preds/`, so every number in the report can be traced back,
4. logs to W&B as `{method}-n{N}-s{seed}` with the full `CFG` as config.

The loop is **resumable**. Runs already in the CSV are skipped, so a Colab disconnect costs only the run in progress. Runs go in `run_order`, $N=1000$ first.

**Q7.1.** Run it.

In [ ]:
METHODS = ["majority", "classical", "probe", "finetune"]


def run_one(method, N, seed):
    idx = SUBSETS[seed][N]
    run = wandb.init(project=CFG["wandb_project"], name=f"{method}-n{N}-s{seed}", reinit=True,
                     config={**CFG, "method": method, "N_target": N, "N": len(idx), "seed": seed},
                     mode=CFG["wandb_mode"], job_type="experiment", group=method)
    t0 = time.time()
    if method == "majority":
        pred = np.full(len(y_test), np.bincount(y_train[idx], minlength=43).argmax())
    elif method == "classical":
        Ftr, Fte = FEATS[CFG["features"]]
        predict, _ = fit_linear(Ftr[idx], y_train[idx], CFG["C"], seed)
        pred = predict(Fte)
    elif method == "probe":
        predict, _ = fit_linear(PROBE[0][idx], y_train[idx], CFG["probe_C"], seed)
        pred = predict(PROBE[1])
    elif method == "finetune":
        model, _ = train_cnn(idx, seed, CFG["lr"], log=wandb.log)
        t_fit = time.time() - t0
        pred = predict_cnn(model, X_test_t)
        del model
    t_total = time.time() - t0
    mf1, per_class = macro_f1(y_test, pred)
    row = dict(method=method, N_target=N, N=len(idx), seed=seed, macro_f1=mf1,
               accuracy=accuracy_score(y_test, pred), seconds=t_total)
    np.save(f"results/preds/{method}_n{N}_s{seed}.npy", pred.astype(np.int16))
    wandb.log({"test/macro_f1": mf1, "test/accuracy": row["accuracy"], "seconds": t_total,
               "test/per_class_f1": wandb.Histogram(per_class)})
    run.summary.update(row)
    run.finish()
    pd.DataFrame([row]).to_csv(CFG["results_csv"], mode="a", index=False,
                               header=not os.path.exists(CFG["results_csv"]))
    return row


done = set()
if os.path.exists(CFG["results_csv"]):
    prev = pd.read_csv(CFG["results_csv"])
    done = set(zip(prev.method, prev.N_target, prev.seed))
todo = [(m, N, s) for N in CFG["run_order"] for s in CFG["seeds"] for m in METHODS
        if (m, N, s) not in done]
print(f"{len(done)} runs already done · {len(todo)} to go")
for m, N, s in todo:
    r = run_one(m, N, s)
    print(f"{m:>9}  N={r['N']:>5}  seed {s}  macro-F1 {r['macro_f1']:.4f}  acc {r['accuracy']:.4f}"
          f"  ({r['seconds']:.0f}s)")

## 8. Results: the data-efficiency curve

**Q8.1.** From `results/results.csv` alone, make the table the report will carry: mean ± standard deviation of macro-F1 and of accuracy across seeds, for every method and size. Save it as `results/summary.csv`.

In [ ]:
R_ = pd.read_csv(CFG["results_csv"])
summary = (R_.groupby(["method", "N_target"])
             .agg(N=("N", "mean"), runs=("seed", "count"),
                  f1_mean=("macro_f1", "mean"), f1_std=("macro_f1", "std"),
                  acc_mean=("accuracy", "mean"), acc_std=("accuracy", "std"))
             .reset_index())
summary.to_csv("results/summary.csv", index=False)
pm = lambda m, s: f"{m:.3f} ± {s:.3f}"
table = summary.assign(cell=[pm(a, b) for a, b in zip(summary.f1_mean, summary.f1_std)]) \
               .pivot(index="N_target", columns="method", values="cell")
print("macro-F1 (mean ± std over seeds)"); print(table[[m for m in METHODS if m in table]].to_string())

**Q8.2.** The figure the project exists to produce: macro-F1 against training-set size for every method on one set of axes, log-scaled $x$, error bars of ±1 standard deviation across seeds. Then a second panel with the **paired gap**, fine-tune minus classical, computed per seed (both methods saw the same images at a given seed, so the difference is paired) and then averaged.

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.5))
style = dict(majority=("0.6", ":"), classical=("C0", "-"), probe=("C2", "--"), finetune=("C3", "-"))
label = dict(majority="majority class", classical=f"MR8 filter bank ({CFG['features']}) + logistic",
             probe="ResNet-18 frozen + logistic", finetune="ResNet-18 fine-tuned")
for m in METHODS:
    s = summary[summary.method == m].sort_values("N")
    if len(s):
        a1.errorbar(s.N, s.f1_mean, yerr=s.f1_std, marker="o", capsize=3,
                    color=style[m][0], ls=style[m][1], label=label[m])
a1.set_xscale("log"); a1.set_xlabel("labelled training images, N (log scale)")
a1.set_ylabel("test macro-F1"); a1.set_ylim(0, 1); a1.grid(alpha=0.3); a1.legend(fontsize=8)
a1.set_title("Data-efficiency curve (GTSRB test, 12,630 images)")

wide = R_.pivot_table(index=["N_target", "seed"], columns="method", values="macro_f1")
for other, col in [("classical", "C3"), ("probe", "C2")]:
    if {"finetune", other} <= set(wide.columns):
        gap = (wide["finetune"] - wide[other]).groupby("N_target").agg(["mean", "std"])
        n_actual = summary[summary.method == "finetune"].set_index("N_target").N.loc[gap.index]
        a2.errorbar(n_actual, gap["mean"], yerr=gap["std"], marker="o", capsize=3, color=col,
                    label=f"fine-tuned − {label[other]}")
        gap.to_csv(f"results/gap_finetune_minus_{other}.csv")
a2.axhline(0, color="k", lw=0.8); a2.set_xscale("log")
a2.set_xlabel("labelled training images, N (log scale)"); a2.set_ylabel("Δ macro-F1 (paired by seed)")
a2.set_title("Gap"); a2.grid(alpha=0.3); a2.legend(fontsize=8)
for a in (a1, a2):
    a.set_xticks(CFG["sizes"]); a.set_xticklabels([f"{n:,}" for n in CFG["sizes"]]); a.minorticks_off()
plt.tight_layout(); plt.savefig("figures/data_efficiency.png", dpi=200); plt.show()

**Q8.3.** State the result in one sentence, in the form the proposal promised: *how does the advantage of the pretrained CNN over the classical pipeline change as labelled data decreases?* Then judge the hypothesis ("the gap widens most sharply in the low-data regime") against the gap panel. Supported, contradicted, or not resolvable with three seeds? Use the error bars.

*Answer:*

**Q8.4.** Compare the linear probe with both. Where does it sit between the classical pipeline and the fine-tuned network, and does that change with $N$? What does that say about whether the advantage comes from *ImageNet features* or from *fine-tuning*?

*Answer:*

**Q8.5.** Does accuracy tell the same story as macro-F1? Find the size at which the two metrics disagree most about the classical pipeline, and explain the disagreement from the class counts of Q1.1.

In [ ]:
acc_tab = summary.assign(cell=[pm(a, b) for a, b in zip(summary.acc_mean, summary.acc_std)]) \
                 .pivot(index="N_target", columns="method", values="cell")
print("accuracy (mean ± std over seeds)"); print(acc_tab[[m for m in METHODS if m in acc_tab]].to_string())
d = summary.assign(acc_minus_f1=summary.acc_mean - summary.f1_mean)
print(d.pivot(index="N_target", columns="method", values="acc_minus_f1").round(3).to_string())

*Answer:*

## 9. Cost

The proposal promised to report model size and the memory, cost and latency trade-off next to accuracy.

**Q9.1.** For the classical pipeline, the probe and the fine-tuned network, report the parameter count and size in MB (float32), test-time latency per image on CPU at batch 1 and throughput on the GPU at batch 512, and the mean training wall-clock at $N=1000$ and $N=5000$ from `results.csv`.

In [ ]:
def timeit(fn, reps):
    fn()  # warm-up
    if DEV == "cuda": torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(reps): fn()
    if DEV == "cuda": torch.cuda.synchronize()
    return (time.time() - t0) / reps

n_feat = FEATS[CFG["features"]][0].shape[1]
params = {
    "classical": BANK.size + n_feat * CFG["n_classes"] + CFG["n_classes"] + 2 * n_feat,
    "probe": sum(p.numel() for p in resnet18().parameters()) - 1000 * 513 + 512 * 43 + 43 + 2 * 512,
    "finetune": sum(p.numel() for p in make_resnet().parameters()),
}

# CPU, batch 1: classical = filter bank + pooling + the fitted linear model; CNN = forward pass
idx = SUBSETS[0][1000]
predict_cls, _ = fit_linear(FEATS[CFG["features"]][0][idx], y_train[idx], CFG["C"])
one = X_test[:1]
net_cpu = resnet18(); net_cpu.fc = nn.Linear(512, 43); net_cpu.eval()
x_cpu = torch.randn(1, 3, CFG["cnn_res"], CFG["cnn_res"])
with torch.no_grad():
    lat = {"classical": timeit(lambda: predict_cls(filter_features(one, CFG["features"], dev="cpu")), 50),
           "finetune": timeit(lambda: net_cpu(x_cpu), 20)}
lat["probe"] = lat["finetune"]

gpu_tp = {}
if DEV == "cuda":
    xb = X_test[:512]
    gpu_tp["classical"] = 512 / timeit(lambda: filter_features(xb, CFG["features"]), 5)
    net = make_resnet().eval()
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        gpu_tp["finetune"] = 512 / timeit(lambda: net(to_input(X_test_t[:512])), 5)
    gpu_tp["probe"] = gpu_tp["finetune"]

tt = R_.groupby(["method", "N_target"]).seconds.mean().unstack()
cost = pd.DataFrame({
    "parameters": params,
    "size_MB_fp32": {k: v * 4 / 1e6 for k, v in params.items()},
    "cpu_ms_per_image_b1": {k: v * 1e3 for k, v in lat.items()},
    "gpu_images_per_s_b512": gpu_tp,
    "train_s_N1000": tt.get(1000, pd.Series(dtype=float)).to_dict(),
    "train_s_N5000": tt.get(5000, pd.Series(dtype=float)).to_dict(),
})
cost = cost.drop(index="majority", errors="ignore")
cost.to_csv("results/cost.csv")
print(cost.round(2).to_string())

**Q9.2.** Put Q8 and Q9 together. At the size where the fine-tuned network's advantage is smallest, what does it cost in size and latency? Name a deployment where you would ship the classical pipeline anyway.

*Answer:*

## 10. Where each method fails

A mean over 43 classes hides which classes carry the gap. Six of the classes are speed limits that differ only in their digits. That is a shape distinction a pooled filter bank is badly placed to make, and one a CNN pretrained on ImageNet has never been asked to make.

**Q10.1.** From the saved predictions (seed 0, $N=1000$), plot per-class F1 for the classical pipeline and the fine-tuned network on one axis, sorted by the gap. List the five classes with the largest gap.

In [ ]:
N0, s0 = 1000, 0
pc = {m: macro_f1(y_test, np.load(f"results/preds/{m}_n{N0}_s{s0}.npy"))[1]
      for m in ["classical", "finetune"] if os.path.exists(f"results/preds/{m}_n{N0}_s{s0}.npy")}
if len(pc) == 2:
    order = np.argsort(pc["finetune"] - pc["classical"])[::-1]
    fig, ax = plt.subplots(figsize=(14, 4))
    k = np.arange(CFG["n_classes"])
    ax.bar(k - 0.2, pc["classical"][order], 0.4, label=label["classical"])
    ax.bar(k + 0.2, pc["finetune"][order], 0.4, label=label["finetune"])
    ax.set_xticks(k); ax.set_xticklabels([CLASS_NAMES[i] for i in order], rotation=90, fontsize=7)
    ax.set_ylabel("test F1"); ax.set_title(f"per-class F1, N={N0}, seed {s0}, sorted by gap")
    ax.legend(); plt.tight_layout(); plt.savefig("figures/per_class_f1.png", dpi=150); plt.show()
    for i in order[:5]:
        print(f"{CLASS_NAMES[i]:>24}: classical {pc['classical'][i]:.3f} · fine-tuned {pc['finetune'][i]:.3f}"
              f" · train images at N={N0}: {np.bincount(y_train[SUBSETS[s0][N0]], minlength=43)[i]}")

**Q10.2.** Show the confusion matrix restricted to the nine speed-limit classes (0–8) for both methods, row-normalised. Which confusions does the classical pipeline make that the network does not? Show three misclassified test images for the worst confusion.

In [ ]:
SL = list(range(9))
if len(pc) == 2:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5), gridspec_kw=dict(width_ratios=[1, 1, 0.9]))
    for ax, m in zip(axes, ["classical", "finetune"]):
        p = np.load(f"results/preds/{m}_n{N0}_s{s0}.npy")
        cm = confusion(y_test, p)[np.ix_(SL, SL)].astype(float)
        cm /= np.maximum(cm.sum(1, keepdims=True), 1)
        ax.imshow(cm, vmin=0, vmax=1, cmap="Blues")
        for i in SL:
            for j in SL:
                ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center", fontsize=6,
                        color="w" if cm[i, j] > 0.5 else "k")
        ax.set_xticks(SL); ax.set_yticks(SL)
        ax.set_xticklabels([CLASS_NAMES[i] for i in SL], rotation=90, fontsize=7)
        ax.set_yticklabels([CLASS_NAMES[i] for i in SL], fontsize=7)
        ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(label[m], fontsize=9)
    p = np.load(f"results/preds/classical_n{N0}_s{s0}.npy")
    cm = confusion(y_test, p)[np.ix_(SL, SL)]; np.fill_diagonal(cm, 0)
    ti, pj = np.unravel_index(cm.argmax(), cm.shape)
    wrong = np.where((y_test == ti) & (p == pj))[0][:3]
    axes[2].imshow(np.concatenate([X_test[i] for i in wrong], 1) if len(wrong) else np.zeros((8, 8)))
    axes[2].set_title(f"classical: true {CLASS_NAMES[ti]} → predicted {CLASS_NAMES[pj]}", fontsize=9)
    axes[2].axis("off")
    plt.tight_layout(); plt.savefig("figures/speed_limit_confusion.png", dpi=150); plt.show()

*Answer:*

**Q10.3.** What would you do next? Name one experiment that would most sharpen the conclusion (not just "more data" or "more seeds"), say what result would change your mind, and estimate what it costs on a free Colab GPU.

*Answer:*

## 11. W&B runs

Paste links below. At minimum: the tuning run of Section 6, and one W&B *group* view per method from Section 7 (W&B groups the 15 runs of each method). Each run's config already carries the full `CFG`: method, $N$, seed, features, $C$, learning rate, steps, batch, resolution and the validation split seed.

*W&B run links:*

## 12. Before you submit

- [ ] **Runtime ▸ Restart session and run all** on a GPU runtime, from a clean checkout. `results/results.csv` is regenerated end to end (delete it first, or the resume logic skips everything).
- [ ] Every *Answer:* cell is filled in, in your own words. Every member can explain every cell.
- [ ] Every number in the report appears in `results/summary.csv`, `results/gap_*.csv`, `results/cost.csv` or `results/tuning.csv`, and every figure is in `figures/`.
- [ ] Figures have labelled axes and units. The $x$-axis of the main plot is labelled *log scale*.
- [ ] The W&B links in Section 11 open, and their configs match `CFG`.
- [ ] `README.md` says how to set up, get the data and run, and the repo has `requirements.txt`.
- [ ] Q1.1 confirms 39,209 / 12,630 images (Kaggle source). If the torchvision fallback was used, the report says so and gives its count.